# Rebuild HotpotQA Span Scan Cache

Set `NUM_SAMPLES` and run the cells below to rebuild the HotpotQA phrase/token span scan cache with the current extraction rules. The output paths intentionally keep the original `qwen_scan_v1_all` names, so this notebook overwrites the existing cache files even when `NUM_SAMPLES` is not all samples.

In [1]:
from pathlib import Path
import shlex
import subprocess

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

# Change this value for the experiment size. Use None for all HotpotQA samples.
NUM_SAMPLES = 500

# Fixed output names. Keep these values if you want to overwrite the original cache files.
CACHE_TAG = "qwen_scan_v1"
OUTPUT_SAMPLE_TAG = "all"

PYTHON_EXECUTABLE = "/home/xiaoyue/anaconda3/envs/llm_graph/bin/python"
HOTPOT_PATH = REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json"
CACHE_DIR = REPO_ROOT / "hotpot_QA_qwen_scan_cache"
SCRIPT_PATH = REPO_ROOT / "prepare_hotpot_span_scan_store.py"

SPACY_MODEL = "en_core_web_lg"
NLP_BATCH_SIZE = 32
DISCARD_NO_WORD = False

# These must stay True to overwrite the original fixed-name caches.
FORCE = True
FORCE_DOCUMENTS = True

DOCUMENTS_CACHE_PATH = CACHE_DIR / f"hotpot_documents_{CACHE_TAG}_{OUTPUT_SAMPLE_TAG}.pkl"
SAMPLES_CACHE_PATH = CACHE_DIR / f"hotpot_samples_{CACHE_TAG}_{OUTPUT_SAMPLE_TAG}.pkl"
SCAN_STORE_PATH = CACHE_DIR / f"hotpot_phrase_token_scan_store_{CACHE_TAG}_{OUTPUT_SAMPLE_TAG}.pkl"

print(f"repo: {REPO_ROOT}")
print(f"num_samples: {NUM_SAMPLES}")
print(f"documents cache: {DOCUMENTS_CACHE_PATH}")
print(f"samples cache: {SAMPLES_CACHE_PATH}")
print(f"scan store: {SCAN_STORE_PATH}")

repo: /home/xiaoyue/LiteSemRAG
num_samples: 500
documents cache: /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_documents_qwen_scan_v1_all.pkl
samples cache: /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_samples_qwen_scan_v1_all.pkl
scan store: /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl


## Build Cache

This cell runs the repository script with `--force`, so the fixed output files above are replaced.

In [2]:
cmd = [
    PYTHON_EXECUTABLE,
    "-u",
    str(SCRIPT_PATH),
    "--hotpot-path",
    str(HOTPOT_PATH),
    "--cache-dir",
    str(CACHE_DIR),
    "--cache-tag",
    CACHE_TAG,
    "--output-sample-tag",
    OUTPUT_SAMPLE_TAG,
    "--spacy-model",
    SPACY_MODEL,
    "--nlp-batch-size",
    str(NLP_BATCH_SIZE),
]

if NUM_SAMPLES is not None:
    cmd.extend(["--num-samples", str(NUM_SAMPLES)])
if DISCARD_NO_WORD:
    cmd.append("--discard-no-word")
if FORCE:
    cmd.append("--force")
if FORCE_DOCUMENTS:
    cmd.append("--force-documents")

print("Running:")
print(" ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT)
if result.returncode != 0:
    raise RuntimeError(f"cache rebuild failed with exit code {result.returncode}")

Running:
/home/xiaoyue/anaconda3/envs/llm_graph/bin/python -u /home/xiaoyue/LiteSemRAG/prepare_hotpot_span_scan_store.py --hotpot-path /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json --cache-dir /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache --cache-tag qwen_scan_v1 --output-sample-tag all --spacy-model en_core_web_lg --nlp-batch-size 32 --num-samples 500 --force --force-documents
Building HotpotQA document cache from /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Saved documents: /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_documents_qwen_scan_v1_all.pkl
Saved samples: /home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_samples_qwen_scan_v1_all.pkl
Loading spaCy model: en_core_web_lg
Scanning 4937 documents with current extraction rules
Processed 500/4937 documents
Processed 1000/4937 documents
Processed 1500/4937 documents
Processed 2000/4937 documents
Processed 2500/4937 documents
Processed 3000/4937 documents

## Inspect Cache

In [3]:
import pickle
from text_processing import normalize_text

CHECK_TERMS = ["space", "director"]

with SCAN_STORE_PATH.open("rb") as handle:
    store = pickle.load(handle)

print("config:")
print(store.get("config"))
print("\nstats:")
print(store.get("stats"))
print("\nterm counts:")
for term in CHECK_TERMS:
    records = store["index"].get(normalize_text(term), [])
    print(f"{term}: {len(records)}")

config:
{'cache_tag': 'qwen_scan_v1', 'cache_dir': '/home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache', 'documents_cache_path': '/home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_documents_qwen_scan_v1_all.pkl', 'samples_cache_path': '/home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_samples_qwen_scan_v1_all.pkl', 'scan_store_path': '/home/xiaoyue/LiteSemRAG/hotpot_QA_qwen_scan_cache/hotpot_phrase_token_scan_store_qwen_scan_v1_all.pkl', 'spacy_model_name': 'en_core_web_lg', 'discard_no_word': False, 'min_tokens': 2, 'scan_only': True, 'span_extractor': 'text_processing.extract_important_spans', 'tokenizer': 'LiteSemRAG hyphen-preserving spaCy tokenizer'}

stats:
{'num_documents': 4937, 'num_unique_terms': 60797, 'num_phrase_occurrences': 77703, 'num_token_occurrences': 93649, 'num_total_occurrences': 171352}

term counts:
space: 78
director: 235
